In [16]:
import json
import pickle
from functools import lru_cache
from google import genai
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import time
from supabase import create_client, Client

In [17]:
supabase_url = os.environ.get("SUPABASE_URL")
supabase_key = os.environ.get("SUPABASE_KEY")

supabase: Client = create_client(supabase_url, supabase_key)

In [2]:
SCHEMA_PATH = "../schema/schema_metadata.json"
CACHE_PATH = "../schema/embeddings_cache.pkl"

In [5]:

KEY = os.environ["GOOGLE_KEY"]

client = genai.Client(api_key=KEY)

In [3]:
@lru_cache(maxsize=1)
def _load_resources():
    """Carga schema, embeddings y modelo una sola vez (cacheado en memoria)."""
    with open(SCHEMA_PATH, encoding="utf-8") as f:
        schema = json.load(f)

    with open(CACHE_PATH, "rb") as f:
        cache = pickle.load(f)

    model = SentenceTransformer(cache["model_name"])

    tables_by_name = {t["name"]: t for t in schema["tables"]}

    return {
        "tables_by_name": tables_by_name,
        "table_names": cache["table_names"],
        "embeddings": cache["embeddings"],
        "model": model,
    }

def _cosine_similarity(query_vec: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    query_norm = query_vec / np.linalg.norm(query_vec)
    matrix_norm = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix_norm @ query_norm


def get_relevant_tables(user_prompt: str, top_k: int = 4) -> list[dict]:
    """
    Devuelve la metadata completa (incluyendo columnas) de las top_k tablas/vistas
    más relevantes para el prompt del usuario, según similitud de embeddings.
    """
    resources = _load_resources()

    query_vec = resources["model"].encode(user_prompt, convert_to_numpy=True)
    similarities = _cosine_similarity(query_vec, resources["embeddings"])

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        table_name = resources["table_names"][idx]
        table_meta = resources["tables_by_name"][table_name]
        results.append({
            "name": table_name,
            "similarity": float(similarities[idx]),
            "metadata": table_meta,
        })

    return results


def format_schema_for_prompt(relevant_tables: list[dict]) -> str:
    """
    Convierte la metadata de las tablas seleccionadas en texto para inyectar
    en el prompt del LLM (schema injection).
    """
    blocks = []
    for entry in relevant_tables:
        meta = entry["metadata"]
        lines = [f"### Table/view: {meta['name']}", f"Description: {meta['description']}"]

        table_note = meta.get("note")
        if table_note:
            lines.append(f"Note: {table_note}")

        lines.append("Columns:")

        for col_name, col_info in meta["columns"].items():
            if isinstance(col_info, dict):
                desc = col_info.get("description", "")
                line = f"  - {col_name}: {desc}"
                if "enum_values" in col_info:
                    values = ", ".join(str(v) for v in col_info["enum_values"])
                    line += f" [posible values: {values}]"
                if "note" in col_info:
                    line += f" (Note: {col_info['note']})"
            else:
                line = f"  - {col_name}: {col_info}"
            lines.append(line)

        blocks.append("\n".join(lines))

    return "\n\n".join(blocks)

In [ ]:
prompts = [
    "Dame el top 10 de los jugadores con más goles en la historia de los mundiales, incluyendo su nombre, país y cantidad de goles.",
    "Dame los 10 directores técnicos con más mundiales dirigidos",
    "Dame el plantel de Argentina en 1990",
    "Dame el top 10 de los jugadores con más partidos jugados en la historia de los mundiales, incluyendo su nombre, país y cantidad de partidos.",
    "Dame el top 10 de los jugadores con más tarjetas amarillas en la historia de los mundiales",
    "¿Cuál fue la pelota oficial del mundial de 1994?",
    "Jugadores con más goles de penal",
    "Cuantos partidos jugó Diego Maradona en mundiales y cuantos goles hizo",
    "Cantidad de arbitros por confederación en la historia de los mundiales",
    "Dame el top 10 de los jugadores con más tarjetas rojas en la historia de los mundiales",
    "¿Como le fua a Alemania en cada mundial?"
    ]

injections = []

for prompt in prompts:
    relevant = get_relevant_tables(prompt, top_k=33) # toda la db (33 tablas), para probar que tanto puede manejar el LLM
    injections.append(format_schema_for_prompt(relevant))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5018.74it/s]


In [8]:
GENERAL_SQL_RULES = """
Reglas generales importantes a tener en cuenta al generar la consulta:

1. Entidades históricas divididas: algunos países aparecen como múltiples filas 
   distintas en `teams` (team_id distintos) debido a cambios políticos históricos:
   - Alemania: 'West Germany' y 'East Germany' (1954-1990) y 'Germany' (1994-presente)
   - URSS/Rusia: 'Soviet Union' (hasta 1990) y 'Russia' (desde 1994)
   - Yugoslavia: 'Yugoslavia' (hasta 1992), 'Serbia and Montenegro' (1992-2006), 'Serbia' (desde 2006), 'Croatia', etc.
   - Czechoslovakia: 'Czechoslovakia' (hasta 1992), 'Czech Republic' (desde 1994), 'Slovakia' (desde 1994)
   - Considerar otros casos que puedan surgir en los datos...

   Esto puede causar dos problemas si no se maneja con cuidado:
   a) Al hacer GROUP BY team_id/team_name sobre estadísticas de carrera de un 
      jugador/DT, su total puede quedar DIVIDIDO en dos filas si jugó bajo ambas 
      entidades (ej. Lothar Matthäus con Alemania Occidental y Alemania), 
      pudiendo hacerlo desaparecer de un ranking top-N.
   b) Al hacer JOIN de una vista de estadísticas de carrera (ya agregada por 
      jugador) con player_appointments/team para obtener el nombre del equipo, 
      un jugador con apariciones bajo ambas entidades genera FILAS DUPLICADAS 
      con el mismo total.
   
   Si la pregunta pide totales de carrera de un jugador/DT junto con su país,
   preferí tomar UN equipo representativo por persona (ej. usando 
   DISTINCT ON (player_id) ordenado por tournament_id DESC para tomar el más 
   reciente) en vez de agrupar o unir de forma que divida o duplique el total.
"""

In [9]:
sql_queries = []

for prompt, injection in zip(prompts, injections):
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=f"""
        rol: especialista en PostgreSQL,
        
        prompt del usuario: 
        {prompt}
        
        tablas disponibles:
        {injection}
        
        {GENERAL_SQL_RULES}
        
        instrucciones: Genera la consulta SQL que responda a la pregunta del prompt 
        usando las tablas disponibles. No agregues explicaciones ni comentarios, 
        solo la consulta SQL. Devolvé únicamente el código SQL, sin backticks ni bloques de markdown.No cometas errores.
        """
    )
    time.sleep(10)
    sql_queries.append(response.text)

In [10]:
for prompt, query in zip(prompts, sql_queries):
    print(f"Prompt: {prompt}\nSQL Query: {query}\n{'-'*80}")

Prompt: Dame el top 10 de los jugadores con más goles en la historia de los mundiales, incluyendo su nombre, país y cantidad de goles.
SQL Query: SELECT
    TRIM(CONCAT(v.given_name, ' ', v.family_name)) AS nombre,
    t.team_name AS pais,
    v.total_goals AS cantidad_de_goles
FROM v_player_stats_career v
JOIN (
    SELECT DISTINCT ON (player_id) player_id, team_id
    FROM player_appointments
    ORDER BY player_id, tournament_id DESC
) pa ON v.player_id = pa.player_id
JOIN teams t ON pa.team_id = t.team_id
ORDER BY v.total_goals DESC
LIMIT 10
--------------------------------------------------------------------------------
Prompt: Dame el top 10 de los jugadores con más partidos jugados en la historia de los mundiales, incluyendo su nombre, país y cantidad de partidos.
SQL Query: SELECT 
    TRIM(CONCAT(p.given_name, ' ', p.family_name)) AS nombre_jugador,
    t.team_name AS pais,
    psc.matches_played
FROM v_player_stats_career psc
JOIN players p ON psc.player_id = p.player_id
JOIN